# Translation Error Annotation Agreement Analysis

This notebook analyzes inter-annotator agreement on translation error annotations
following a custom MQM (Multidimensional Quality Metrics) typology.

In this notebook, we calculate:
- Visualizations of annotator agreement on the text (span overlap)
- Exact matching for span boundaries
- F1 for partial credit on spans
- Cohen's Kappa for category, subcategory, and impact level agreement

## 1. Project Config

In [35]:
# ── PROJECT CONFIG ────────────────────────────────────────────────────────────
# Edit this cell when starting a new project or a new round of annotations.
#
# ANNOTATION_DIR  → folder containing one or more Label Studio JSON exports.
#                   Drop new export files here; the notebook picks them all up.
# PROJECT_NAME    → used in report titles and output filenames.
# REPORT_DATE     → date label for this round (YYYY-MM-DD).
# OUTPUT_DIR      → where the HTML report will be written.
# ──────────────────────────────────────────────────────────────────────────────

PROJECT_NAME   = "QUALITY REVIEWS"
ANNOTATION_DIR = "./data"
OUTPUT_DIR     = "./reports"
REPORT_DATE    = "2026-03-19"   # update each round

## 2. Imports

In [36]:
import json
import os
import glob
import datetime
import re
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print(f"pandas     {pd.__version__}")
print(f"numpy      {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"seaborn    {sns.__version__}")

plt.style.use('ggplot')
sns.set_palette("colorblind")

pandas     2.2.3
numpy      2.2.6
matplotlib 3.10.0
seaborn    0.13.2


## 3. Data Loading and Parsing

Label Studio exports a JSON file per project. Each file is a list of **tasks**
(one task = one text). Each task contains an `annotations` list (one entry per
evaluator). The annotations' `result` array holds all span-level labels and
document-level ratings as separate items distinguished by `from_name`.

The two functions below handle the full pipeline:

1. `load_annotation_files` — globs all `.json` files from `ANNOTATION_DIR`
2. `parse_label_studio_export` — reads every task in every file and groups
   annotations by **task ID** (Label Studio's own unique integer per text).
   This works across single- and multi-file exports automatically.

Field mapping from the `result` array:

| `from_name`               | Maps to            |
|---------------------------|--------------------|
| `label`                   | span `labels`      |
| `subcategories`           | span `subcategory` |
| `impact`                  | span `impact`      |
| `comments`                | span `comments`    |
| `overall_correspondence`  | doc-level rating   |
| `overall_readability`     | doc-level rating   |
| `document_issues`         | doc-level text     |
| `correspondence_comments` | doc-level text     |
| `readability_comments`    | doc-level text     |
| `type: relation`          | skipped (future)   |

In [37]:
def load_annotation_files(annotation_dir):
    """
    Return a sorted list of all .json file paths in annotation_dir.
    Raises FileNotFoundError with a clear message if the dir or files are missing.
    """
    if not os.path.isdir(annotation_dir):
        raise FileNotFoundError(f"Annotation directory not found: {annotation_dir}")
    paths = sorted(glob.glob(os.path.join(annotation_dir, "*.json")))
    if not paths:
        raise FileNotFoundError(f"No JSON files found in: {annotation_dir}")
    print(f"Found {len(paths)} JSON file(s):")
    for p in paths:
        print(f"  {os.path.basename(p)}")
    return paths


def parse_result_array(result):
    """
    Flatten a Label Studio annotation's result array into a structured dict.

    Span items share a region ID — we group them before building enriched_labels.
    Document-level fields are collected separately.
    Relation items (from_name=None) are silently skipped.
    """
    regions = {}  # region_id → accumulated field values

    doc_fields = {
        "overall_correspondence":  None,
        "overall_readability":     None,
        "document_issues":         None,
        "correspondence_comments": None,
        "readability_comments":    None,
    }

    for item in result:
        from_name = item.get("from_name")
        if from_name is None:
            continue  # relation items

        value     = item.get("value", {})
        region_id = item.get("id")

        # ── Span-level fields ──────────────────────────────────────────────
        if from_name == "label":
            if region_id not in regions:
                regions[region_id] = {
                    "start": value.get("start"),
                    "end":   value.get("end"),
                    "text":  value.get("text", ""),
                }
            regions[region_id]["labels"] = value.get("labels", [])

        elif from_name == "impact":
            if region_id not in regions:
                regions[region_id] = {}
            choices = value.get("choices", [])
            regions[region_id]["impact"] = choices[0] if choices else None

        elif from_name == "subcategories":
            if region_id not in regions:
                regions[region_id] = {}
            choices = value.get("choices", [])
            regions[region_id]["subcategory"] = choices[0] if choices else None

        elif from_name == "comments":
            if region_id not in regions:
                regions[region_id] = {}
            texts = value.get("text", [])
            regions[region_id]["comments"] = texts[0] if texts else None

        # ── Document-level fields ──────────────────────────────────────────
        elif from_name == "overall_correspondence":
            doc_fields["overall_correspondence"] = value.get("rating")

        elif from_name == "overall_readability":
            doc_fields["overall_readability"] = value.get("rating")

        elif from_name in ("document_issues", "correspondence_comments", "readability_comments"):
            texts = value.get("text", [])
            doc_fields[from_name] = texts[0] if texts else None

    # Build enriched_labels — only regions that have a label entry
    enriched_labels = []
    for region_id, region in regions.items():
        if "labels" not in region:
            continue
        enriched_labels.append({
            "start":       region.get("start"),
            "end":         region.get("end"),
            "text":        region.get("text", ""),
            "labels":      region.get("labels", []),
            "subcategory": region.get("subcategory"),
            "impact":      region.get("impact"),
            "comments":    region.get("comments"),
        })

    return {"enriched_labels": enriched_labels, **doc_fields}


def parse_label_studio_export(file_paths):
    """
    Load all Label Studio JSON exports and return:
        task_id_to_annotations  dict[int, list[dict]]

    Each annotation dict contains:
        task_id, annotation_id, annotator_id,
        text, lead_time, lead_time_minutes,
        created_at, updated_at, review_time, review_time_minutes,
        enriched_labels, overall_correspondence, overall_readability,
        document_issues, correspondence_comments, readability_comments,
        missing_fields
    """
    task_id_to_annotations = {}

    for file_path in file_paths:
        print(f"\nReading: {os.path.basename(file_path)}")
        with open(file_path, "r", encoding="utf-8") as f:
            tasks = json.load(f)
        print(f"  {len(tasks)} task(s)")

        for task in tasks:
            task_id = task["id"]
            text    = task["data"]["text"]

            if task_id not in task_id_to_annotations:
                task_id_to_annotations[task_id] = []

            for raw_ann in task.get("annotations", []):
                if raw_ann.get("was_cancelled"):
                    continue

                parsed = parse_result_array(raw_ann.get("result", []))

                review_time_seconds = None
                try:
                    created = datetime.datetime.fromisoformat(
                        raw_ann["created_at"].replace("Z", "+00:00"))
                    updated = datetime.datetime.fromisoformat(
                        raw_ann["updated_at"].replace("Z", "+00:00"))
                    review_time_seconds = (updated - created).total_seconds()
                except (KeyError, ValueError, TypeError):
                    pass

                missing_fields = [
                    f for f in ("overall_correspondence", "overall_readability",
                                "document_issues", "correspondence_comments",
                                "readability_comments")
                    if parsed.get(f) is None
                ]

                annotation = {
                    "task_id":             task_id,
                    "annotation_id":       raw_ann["id"],
                    "annotator_id":        raw_ann["completed_by"],
                    "text":                text,
                    "lead_time":           raw_ann.get("lead_time"),
                    "lead_time_minutes":   round(raw_ann.get("lead_time", 0) / 60, 2),
                    "created_at":          raw_ann.get("created_at"),
                    "updated_at":          raw_ann.get("updated_at"),
                    "review_time":         review_time_seconds,
                    "review_time_minutes": (
                        round(review_time_seconds / 60, 2)
                        if review_time_seconds is not None else None
                    ),
                    "missing_fields":      missing_fields,
                    **parsed,
                }
                task_id_to_annotations[task_id].append(annotation)

    return task_id_to_annotations

In [38]:
annotation_files          = load_annotation_files(ANNOTATION_DIR)
task_id_to_annotations    = parse_label_studio_export(annotation_files)

# Quick summary
total_annotations = sum(len(v) for v in task_id_to_annotations.values())
print(f"\n{'='*50}")
print(f"Tasks:       {len(task_id_to_annotations)}")
print(f"Annotations: {total_annotations}")
for task_id, anns in task_id_to_annotations.items():
    spans = sum(len(a['enriched_labels']) for a in anns)
    print(f"  Task {task_id}: {len(anns)} annotator(s), {spans} spans total")

Found 1 JSON file(s):
  Algo-MentalHealthProblems_esp-419_1_parsed.json

Reading: Algo-MentalHealthProblems_esp-419_1_parsed.json
  1 task(s)

Tasks:       1
Annotations: 2
  Task 32: 2 annotator(s), 83 spans total


## 4. Exploratory Data Analysis

### Visualizing Overlaps in Translation Error Annotations - Code

This code produces a visualization of the annotated text, where annotations are highlighted and text highlighted in darker colors reflects that more annotators agreed that the span constituted an error. Hovering over the spans produces a pop up that states the number of annotators who marked that span, plus their categorization (label, subcategory, impact) and comments.

In [39]:
def create_span_overlap_visualization(doc_id, task_id_to_annotations):
    """
    Create a visualization showing text with highlights based on annotation span overlap.

    Args:
        doc_id: The document ID to visualize
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations

    Returns:
        HTML string with the visualization
    """
    # Get all annotations for this document
    annotations = task_id_to_annotations.get(doc_id, [])

    if not annotations:
        return f"<p>No annotations found for document ID {doc_id}</p>"

    # Get the text content from the first annotation (all should have the same text)
    text = annotations[0].get('text', '')

    if not text:
        return f"<p>No text content found for document ID {doc_id}</p>"

    # Collect all spans from all annotators
    all_spans = []
    annotator_ids = set()

    for annotation in annotations:
        annotator = annotation.get('annotator_id')
        annotator_ids.add(annotator)

        enriched_labels = annotation.get('enriched_labels', [])
        for label in enriched_labels:
            start = label.get('start', 0)
            end = label.get('end', 0)

            # Skip invalid spans
            if start >= end or start < 0 or end > len(text):
                continue

            all_spans.append({
                'annotator': annotator,
                'start': start,
                'end': end,
                'text': label.get('text', ''),
                'labels': label.get('labels', []),
                'subcategory': label.get('subcategory', ''),
                'impact': label.get('impact', ''),
                'comments': label.get('comments', '')
            })

    # Create an array to track spans at each character position
    # Instead of just counts, we'll store the full span information at each position
    char_spans = [[] for _ in range(len(text))]

    # Mark each span in the character spans array
    for span in all_spans:
        for i in range(span['start'], span['end']):
            if i < len(char_spans):
                char_spans[i].append(span)

    # Create a count array for the gradient calculations
    char_count = [len(spans) for spans in char_spans]

    # Find maximum overlap for normalization
    max_overlap = max(char_count) if char_count else 1

    # Generate HTML with spans for highlighting
    html_parts = []
    i = 0
    while i < len(text):
        if char_count[i] > 0:
            # Start of a highlighted region
            overlap = char_count[i]
            start_i = i

            # Find where this level of overlap ends
            while i < len(text) and char_count[i] == overlap:
                i += 1

            # Calculate highlight intensity (0 to 1)
            intensity = overlap / max_overlap

            # Generate color (lighter blue palette for better readability)
            r = int(220 - (100 * intensity))
            g = int(220 - (100 * intensity))
            b = int(255 - (35 * intensity))

            bg_color = f"rgb({r}, {g}, {b})"

            # Create detailed tooltip that shows all annotations for this span
            tooltip_content = []

            # Get all unique spans in this region
            span_positions = range(start_i, i)
            region_spans = set()
            for pos in span_positions:
                for span in char_spans[pos]:
                    # Use a tuple of annotator+start+end as a unique identifier for the span
                    span_id = (span['annotator'], span['start'], span['end'])
                    region_spans.add(span_id)

            # Get the full span objects from their IDs
            unique_spans = []
            for span_id in region_spans:
                for pos in span_positions:
                    for span in char_spans[pos]:
                        if (span['annotator'], span['start'], span['end']) == span_id and span not in unique_spans:
                            unique_spans.append(span)
                            break

            # Build tooltip content for each unique span
            for span in unique_spans:
                span_info = f"Annotator: {span['annotator']}"

                # Add labels if available
                if span['labels']:
                    labels_str = ', '.join(span['labels'])
                    span_info += f" | Labels: {labels_str}"

                # Add subcategory if available
                if span['subcategory']:
                    span_info += f" | Subcategory: {span['subcategory']}"

                # Add impact if available
                if span.get('impact'):
                    span_info += f" | Impact: {span['impact']}"

                # Add comments if available - tried both comments and comment fields
                if 'comments' in span and span['comments']:
                    # Escape any quotes in the comments to prevent breaking the HTML
                    comments = span['comments'].replace('"', '&quot;').replace("'", "&#39;")
                    span_info += f" | Comments: {comments}"

                tooltip_content.append(span_info)

            # Join all tooltip content with line breaks
            tooltip = f"{overlap} annotator(s)&#10;&#10;" + "&#10;".join(tooltip_content)

            # Create a span with appropriate styling and tooltip
            span_text = text[start_i:i].replace('\n', '<br>')
            html_parts.append(
                f'<span style="background-color: {bg_color};" '
                f'title="{tooltip}">{span_text}</span>'
            )
        else:
            # Non-highlighted text
            start_i = i

            # Find where non-highlighted region ends
            while i < len(text) and char_count[i] == 0:
                i += 1

            # Add the non-highlighted text
            span_text = text[start_i:i].replace('\n', '<br>')
            html_parts.append(span_text)

    # Balanced HTML structure with "just right" spacing
    html = f"""
    <div style="font-family: Arial, sans-serif; line-height: 1.3; margin: 0; padding: 0;">
        <h3 style="margin: 0 0 4px 0;">Document ID: {doc_id}</h3>
        <div style="margin: 0 0 6px 0; white-space: nowrap;">
            <span><strong>Number of unique annotators:</strong> {len(annotator_ids)}</span>
            <span style="margin-left: 10px;"><strong>Total annotations:</strong> {len(all_spans)}</span>
            <span style="margin-left: 10px;"><strong>Maximum overlap:</strong> {max_overlap} annotator{'' if max_overlap == 1 else 's'}</span>
        </div>
        <div style="border: 1px solid #ccc; padding: 10px; margin: 4px 0;">
            {''.join(html_parts)}
        </div>
        <div style="margin: 6px 0 0 0;">
            <p style="margin: 0 0 5px 0;"><strong>Legend:</strong></p>
            <div style="display: flex; flex-wrap: wrap; gap: 5px;">
    """

    # Generate a color legend with centered text
    for i in range(1, max_overlap + 1):
        intensity = i / max_overlap
        r = int(220 - (100 * intensity))
        g = int(220 - (100 * intensity))
        b = int(255 - (35 * intensity))
        bg_color = f"rgb({r}, {g}, {b})"

        html += f"""
            <div style="background-color: {bg_color}; display: flex; align-items: center; justify-content: center; min-width: 100px; height: 40px; text-align: center; margin-right: 5px;">
                {i} annotator{'' if i == 1 else 's'}
            </div>
        """

    html += """
            </div>
        </div>
    </div>
    """

    return html

def visualize_all_documents(task_id_to_annotations, selected_doc_id=None):
    """
    Generate visualizations for all documents or a specific document in the dataset.

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations
        selected_doc_id: Optional specific document ID to visualize

    Returns:
        HTML string with visualizations
    """
    all_html = "<h2 style='margin-bottom: 5px;'>Annotation Overlap Visualization</h2>"

    if selected_doc_id is not None:
        # Visualize only the selected document
        if selected_doc_id in task_id_to_annotations:
            all_html += create_span_overlap_visualization(selected_doc_id, task_id_to_annotations)
        else:
            all_html += f"<p>Document ID {selected_doc_id} not found in the dataset.</p>"
    else:
        # Visualize all documents
        # Sort document IDs to ensure consistent order
        doc_ids = sorted(task_id_to_annotations.keys())

        for i, doc_id in enumerate(doc_ids):
            all_html += create_span_overlap_visualization(doc_id, task_id_to_annotations)
            if i < len(doc_ids) - 1:  # Don't add hr after the last document
                all_html += "<hr style='margin: 12px 0;'>"

    return all_html

# Add the code to display the visualizations
from IPython.display import display, HTML

# Function to display all documents
def display_annotation_overlaps(doc_id=None):
    """
    Display annotation overlap visualizations.

    Args:
        doc_id: Optional specific document ID to visualize
    """
    html_output = visualize_all_documents(task_id_to_annotations, doc_id)
    display(HTML(html_output))

### Visualizing Overlaps in Translation Error Annotations - Visualizations

In [40]:
# Run the visualization on all documents
display_annotation_overlaps()

In [41]:
# OR display a specific document (e.g., document ID 1)
display_annotation_overlaps(doc_id=26)  # replace with a valid document ID from the list above throughout the notebook

### Annotator Data — Code


Here, summaries of annotator data that provide nuance to the analysis are presented, such as lead time and review time.

In [42]:
def summarize_annotation_timing(processed_data, doc_id=None):
    """
    Create a clear tabular summary of annotation timing statistics.

    Args:
        processed_data: Output from preprocess_document_level_data
        doc_id: Optional specific document ID to analyze (None for all documents)
    """
    import pandas as pd
    import numpy as np

    # Create a DataFrame from the processed data
    df = pd.DataFrame(processed_data)

    # Filter by document ID if specified
    if doc_id is not None:
        df = df[df['task_id'] == doc_id]
        title_suffix = f" (Document ID: {doc_id})"
    else:
        title_suffix = " (All Documents)"

    print(f"\n==== ANNOTATION TIMING ANALYSIS{title_suffix} ====\n")

    # Overall statistics
    print("OVERALL TIMING STATISTICS:")
    print("--------------------------")

    for time_type, column in [('Lead Time', 'lead_time_minutes'),
                             ('Review Time', 'review_time_minutes')]:
        data = df[column].dropna()

        if len(data) > 0:
            print(f"\n{time_type} (minutes):")
            print(f"  Number of measurements: {len(data)}")
            print(f"  Median: {data.median():.1f}")
            print(f"  Mean: {data.mean():.1f}")
            print(f"  Q1 (25%): {data.quantile(0.25):.1f}")
            print(f"  Q3 (75%): {data.quantile(0.75):.1f}")
            print(f"  Min: {data.min():.1f}")
            print(f"  Max: {data.max():.1f}")

            # Calculate outliers by 1.5 IQR rule
            q1, q3 = data.quantile(0.25), data.quantile(0.75)
            iqr = q3 - q1
            upper_bound = q3 + 1.5 * iqr

            outliers = data[data > upper_bound]
            if len(outliers) > 0:
                print(f"\n  Outliers (> {upper_bound:.1f} min): {len(outliers)} values")
                print(f"  Outlier values: {', '.join([f'{x:.1f}' for x in sorted(outliers)])}")

    # Per-document analysis if not filtered
    if doc_id is None and 'task_id' in df.columns:
        print("\n\nTIMING STATISTICS BY DOCUMENT:")
        print("-----------------------------")

        doc_ids = sorted(df['task_id'].unique())

        # Create summary table data
        table_data = []
        for current_doc_id in doc_ids:
            doc_df = df[df['task_id'] == current_doc_id]

            # Lead time stats
            lead_times = doc_df['lead_time_minutes'].dropna()
            lead_count = len(lead_times)
            lead_median = lead_times.median() if lead_count > 0 else float('nan')
            lead_q1 = lead_times.quantile(0.25) if lead_count > 0 else float('nan')
            lead_q3 = lead_times.quantile(0.75) if lead_count > 0 else float('nan')
            lead_min = lead_times.min() if lead_count > 0 else float('nan')
            lead_max = lead_times.max() if lead_count > 0 else float('nan')

            # Review time stats
            review_times = doc_df['review_time_minutes'].dropna()
            review_count = len(review_times)
            review_median = review_times.median() if review_count > 0 else float('nan')
            review_q1 = review_times.quantile(0.25) if review_count > 0 else float('nan')
            review_q3 = review_times.quantile(0.75) if review_count > 0 else float('nan')
            review_min = review_times.min() if review_count > 0 else float('nan')
            review_max = review_times.max() if review_count > 0 else float('nan')

            table_data.append({
                'Doc ID': current_doc_id,
                'Lead Count': lead_count,
                'Lead Median': lead_median,
                'Lead Q1-Q3': f"{lead_q1:.1f}-{lead_q3:.1f}" if lead_count > 0 else "N/A",
                'Lead Min-Max': f"{lead_min:.1f}-{lead_max:.1f}" if lead_count > 0 else "N/A",
                'Review Count': review_count,
                'Review Median': review_median,
                'Review Q1-Q3': f"{review_q1:.1f}-{review_q3:.1f}" if review_count > 0 else "N/A",
                'Review Min-Max': f"{review_min:.1f}-{review_max:.1f}" if review_count > 0 else "N/A"
            })

        # Create and format DataFrame
        summary_df = pd.DataFrame(table_data)

        # Format numeric columns
        for col in ['Lead Median', 'Review Median']:
            summary_df[col] = summary_df[col].apply(lambda x: f"{x:.1f}" if not pd.isna(x) else "N/A")

        # Print the table
        print("\nSummary by Document:")
        print(summary_df.to_string(index=False))

        # Print outliers by document
        print("\nOutliers by Document:")
        for current_doc_id in doc_ids:
            doc_df = df[df['task_id'] == current_doc_id]

            print(f"\nDocument {current_doc_id}:")

            # Check for lead time outliers
            lead_times = doc_df['lead_time_minutes'].dropna()
            if len(lead_times) >= 4:  # Need reasonable amount of data for outlier detection
                q1, q3 = lead_times.quantile(0.25), lead_times.quantile(0.75)
                iqr = q3 - q1
                upper_bound = q3 + 1.5 * iqr
                outliers = lead_times[lead_times > upper_bound]

                if len(outliers) > 0:
                    print(f"  Lead Time outliers (> {upper_bound:.1f} min): {len(outliers)} values")
                    print(f"  Values: {', '.join([f'{x:.1f}' for x in sorted(outliers)])}")
                else:
                    print("  Lead Time: No outliers detected")
            else:
                print("  Lead Time: Insufficient data for outlier detection")

            # Check for review time outliers
            review_times = doc_df['review_time_minutes'].dropna()
            if len(review_times) >= 4:
                q1, q3 = review_times.quantile(0.25), review_times.quantile(0.75)
                iqr = q3 - q1
                upper_bound = q3 + 1.5 * iqr
                outliers = review_times[review_times > upper_bound]

                if len(outliers) > 0:
                    print(f"  Review Time outliers (> {upper_bound:.1f} min): {len(outliers)} values")
                    print(f"  Values: {', '.join([f'{x:.1f}' for x in sorted(outliers)])}")
                else:
                    print("  Review Time: No outliers detected")
            else:
                print("  Review Time: Insufficient data for outlier detection")

    # Return timing data if needed for further analysis
    timing_data = {
        'lead_time': df['lead_time_minutes'].dropna().values,
        'review_time': df['review_time_minutes'].dropna().values
    }

    return timing_data

### Annotator Data — Summaries

- **Span completeness**: percentage of spans where each annotator filled in subcategory, impact level, and span-level comment.
- **Document-level completeness**: whether each annotator completed all four document-level fields (correspondence score, correspondence comment, readability score, readability comment).
- **Lead time**: active time to complete the task (as recorded by Label Studio).
- **Review time**: elapsed time from task start to submission.


In [43]:
# Build processed_data from task_id_to_annotations
# This flattens the nested structure into a list of dicts for timing/rating analysis
processed_data = [
    annotation
    for annotations in task_id_to_annotations.values()
    for annotation in annotations
]

def analyze_annotator_distribution(processed_data, doc_id=None):
    """
    Show annotator count and timing summary for a specific task or all tasks.
    """
    df = pd.DataFrame(processed_data)
    if doc_id is not None:
        df = df[df['task_id'] == doc_id]
        print(f"Task {doc_id} — {len(df)} annotator(s)")
    else:
        print(f"All tasks — {len(df)} annotation(s)")

    print(f"  Annotator IDs: {sorted(df['annotator_id'].unique().tolist())}")
    missing = df[df['missing_fields'].apply(len) > 0]
    if len(missing):
        print(f"  {len(missing)} annotation(s) with missing fields:")
        for _, row in missing.iterrows():
            print(f"    Annotator {row['annotator_id']}: {row['missing_fields']}")
    else:
        print("  All annotations complete ✓")

In [44]:
def summarize_annotation_completeness(task_id_to_annotations, doc_id=None):
    """
    Report completeness of span-level and document-level annotations per annotator.

    Span-level checks  : subcategory, impact, comments
    Document-level checks: overall_correspondence (score + comment),
                           overall_readability (score + comment)

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations
        doc_id: Optional specific document ID to filter to (None = all documents)

    Returns:
        DataFrame with one row per (task_id, annotator_id)
    """
    rows = []
    for task_id, annotations in task_id_to_annotations.items():
        if doc_id is not None and task_id != doc_id:
            continue
        for ann in annotations:
            annotator = ann['annotator_id']
            spans = ann.get('enriched_labels', [])
            total_spans = len(spans)

            missing_sub     = sum(1 for s in spans if not s.get('subcategory'))
            missing_impact  = sum(1 for s in spans if not s.get('impact'))
            missing_comment = sum(1 for s in spans if not s.get('comments'))

            rows.append({
                'task_id':                    task_id,
                'annotator_id':               annotator,
                'total_spans':                total_spans,
                'subcategory_%':              round(100 * (total_spans - missing_sub)     / total_spans, 1) if total_spans else None,
                'impact_%':                   round(100 * (total_spans - missing_impact)  / total_spans, 1) if total_spans else None,
                'span_comments_%':            round(100 * (total_spans - missing_comment) / total_spans, 1) if total_spans else None,
                'has_document_issues':        ann.get('document_issues') is not None,
                'has_correspondence_score':   ann.get('overall_correspondence')          is not None,
                'has_correspondence_comment': ann.get('correspondence_comments')  is not None,
                'has_readability_score':      ann.get('overall_readability')             is not None,
                'has_readability_comment':    ann.get('readability_comments')     is not None,
            })

    if not rows:
        print("No data found for the specified filter.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    title_suffix = f" (Document {doc_id})" if doc_id is not None else " (All Documents)"

    # ── Span-level completeness ────────────────────────────────────────────
    print(f"SPAN-LEVEL COMPLETENESS{title_suffix}")
    print("% of spans with field filled in\n")
    span_cols = ['task_id', 'annotator_id', 'total_spans',
                 'subcategory_%', 'impact_%', 'span_comments_%']
    print(df[span_cols].to_string(index=False))

    # Flag annotators below 80% on any span field
    print("\nAnnotators below 80% on any span field:")
    span_num = df[['subcategory_%', 'impact_%', 'span_comments_%']].apply(pd.to_numeric, errors='coerce')
    flagged = df[(span_num < 80).any(axis=1)]
    if len(flagged):
        print(flagged[span_cols].to_string(index=False))
    else:
        print("  None ✓")

    # ── Document-level completeness ────────────────────────────────────────
    print(f"\n\nDOCUMENT-LEVEL COMPLETENESS{title_suffix}\n")
    doc_cols = ['task_id', 'annotator_id', 
                'has_document_issues',
                'has_correspondence_score', 'has_correspondence_comment',
                'has_readability_score',    'has_readability_comment']
    print(df[doc_cols].to_string(index=False))

    # Flag missing document-level fields
    print("\nAnnotators with incomplete document-level assessment:")
    doc_bool = df[['has_document_issues', 'has_correspondence_score', 'has_correspondence_comment',
                   'has_readability_score',    'has_readability_comment']]
    incomplete = df[~doc_bool.all(axis=1)]
    if len(incomplete):
        print(incomplete[doc_cols].to_string(index=False))
    else:
        print("  None ✓")

    return df

# Run for all documents
completeness_df = summarize_annotation_completeness(task_id_to_annotations)


SPAN-LEVEL COMPLETENESS (All Documents)
% of spans with field filled in

 task_id  annotator_id  total_spans  subcategory_%  impact_%  span_comments_%
      32            58           49           93.9      98.0             85.7
      32            61           34          100.0     100.0            100.0

Annotators below 80% on any span field:
  None ✓


DOCUMENT-LEVEL COMPLETENESS (All Documents)

 task_id  annotator_id  has_document_issues  has_correspondence_score  has_correspondence_comment  has_readability_score  has_readability_comment
      32            58                 True                      True                        True                   True                     True
      32            61                 True                      True                       False                   True                    False

Annotators with incomplete document-level assessment:
 task_id  annotator_id  has_document_issues  has_correspondence_score  has_correspondence_comment  has_

In [45]:
# Show annotator distribution for a specific document
analyze_annotator_distribution(processed_data, doc_id=26)

Task 26 — 0 annotator(s)
  Annotator IDs: []
  All annotations complete ✓


In [46]:
# Overall summary for all documents
summarize_annotation_timing(processed_data);


==== ANNOTATION TIMING ANALYSIS (All Documents) ====

OVERALL TIMING STATISTICS:
--------------------------

Lead Time (minutes):
  Number of measurements: 2
  Median: 125.6
  Mean: 125.6
  Q1 (25%): 121.5
  Q3 (75%): 129.8
  Min: 117.4
  Max: 133.9

Review Time (minutes):
  Number of measurements: 2
  Median: 5123.2
  Mean: 5123.2
  Q1 (25%): 2561.6
  Q3 (75%): 7684.8
  Min: 0.0
  Max: 10246.3


TIMING STATISTICS BY DOCUMENT:
-----------------------------

Summary by Document:
 Doc ID  Lead Count Lead Median  Lead Q1-Q3 Lead Min-Max  Review Count Review Median  Review Q1-Q3 Review Min-Max
     32           2       125.6 121.5-129.8  117.4-133.9             2        5123.2 2561.6-7684.8    0.0-10246.3

Outliers by Document:

Document 32:
  Lead Time: Insufficient data for outlier detection
  Review Time: Insufficient data for outlier detection


In [47]:
# Summary for a specific document
summarize_annotation_timing(processed_data, doc_id=26);


==== ANNOTATION TIMING ANALYSIS (Document ID: 26) ====

OVERALL TIMING STATISTICS:
--------------------------


### Error Types Identified - Code

Here, visualizations on the types of errors (label, subcategory, impact) identified are presented.

In [48]:
def visualize_error_types(task_id_to_annotations, doc_id=None, include_subcategories=False):
    """
    Visualize the distribution of error types across annotations.

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations
        doc_id: Optional specific document ID to analyze (None for all documents)
        include_subcategories: Whether to show subcategories (default: False)
    """
    from collections import Counter

    # Filter annotations by document ID if specified
    if doc_id is not None:
        annotations = task_id_to_annotations.get(doc_id, [])
        title_suffix = f" (Document ID: {doc_id})"
    else:
        # Combine all annotations
        annotations = []
        for doc_annotations in task_id_to_annotations.values():
            annotations.extend(doc_annotations)
        title_suffix = " (All Documents)"

    # Extract error labels from all annotations
    labels_data = []
    subcategory_data = []

    for annotation in annotations:
        enriched_labels = annotation.get('enriched_labels', [])

        for label_item in enriched_labels:
            # Extract main error categories
            main_labels = label_item.get('labels', [])

            for main_label in main_labels:
                labels_data.append({
                    'task_id': annotation.get('task_id'),
                    'annotator': annotation.get('annotator_id'),
                    'error_type': main_label,
                    'impact': label_item.get('impact', 'Unknown')
                })

            # Extract subcategory if available
            subcategory = label_item.get('subcategory', 'None')
            if subcategory and subcategory != 'None':
                # Split subcategory (format is typically "CATEGORY: Description")
                subcategory_parts = subcategory.split(':', 1)
                if len(subcategory_parts) > 1:
                    category_code = subcategory_parts[0].strip()
                    description = subcategory_parts[1].strip()
                else:
                    category_code = subcategory
                    description = subcategory

                subcategory_data.append({
                    'task_id': annotation.get('task_id'),
                    'annotator': annotation.get('annotator_id'),
                    'category': category_code,
                    'description': description,
                    'full_subcategory': subcategory,
                    'impact': label_item.get('impact', 'Unknown')
                })

    # Create DataFrames
    labels_df = pd.DataFrame(labels_data)
    subcategory_df = pd.DataFrame(subcategory_data)

    # Print summary statistics
    print(f"Error Type Analysis{title_suffix}")
    print(f"Total annotations analyzed: {len(annotations)}")
    print(f"Total error labels found: {len(labels_df)}")
    print(f"Total error subcategories found: {len(subcategory_df)}")

    # If no error labels were found, return early
    if len(labels_df) == 0:
        print("No error labels found in the selected document(s).")
        return None

    # Count the frequency of each main error type
    main_counts = labels_df['error_type'].value_counts().reset_index()
    main_counts.columns = ['Error Type', 'Count']

    # Sort by count (descending)
    main_counts = main_counts.sort_values('Count', ascending=False)

    # Print main error type distribution
    print("\nMain Error Types Distribution:")
    for i, row in main_counts.iterrows():
        print(f"  {row['Error Type']}: {row['Count']} occurrences")

    # Create bar chart for main error types
    plt.figure(figsize=(12, 6))

    # Use a colorblind-friendly palette
    colors = sns.color_palette("colorblind", len(main_counts))

    # Create the bar chart
    bars = plt.bar(main_counts['Error Type'], main_counts['Count'], color=colors)

    # Add count labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{int(height)}', ha='center', va='bottom')

    # Add labels and title
    plt.xlabel('Error Type', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.title(f'Distribution of Error Types{title_suffix}', fontsize=14)

    # Rotate x-axis labels if there are many categories
    if len(main_counts) > 5:
        plt.xticks(rotation=45, ha='right')

    # Add grid for better readability
    plt.grid(axis='y', linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # Error type distribution by document if no specific doc_id was provided
    if doc_id is None and 'task_id' in labels_df.columns:
        plt.figure(figsize=(14, 8))

        # Count error types by document
        doc_error_counts = labels_df.groupby(['task_id', 'error_type']).size().reset_index(name='count')

        # Create grouped bar chart
        sns.barplot(x='task_id', y='count', hue='error_type', data=doc_error_counts)

        plt.title('Error Types by Document', fontsize=14)
        plt.xlabel('Document ID', fontsize=12)
        plt.ylabel('Count', fontsize=12)
        plt.legend(title='Error Type')
        plt.grid(axis='y', linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.show()

    # If subcategories are requested, visualize them too
    if include_subcategories and len(subcategory_df) > 0:
        # Analyze subcategories
        # Group by category code and get counts
        subcategory_counts = subcategory_df['category'].value_counts().reset_index()
        subcategory_counts.columns = ['Category', 'Count']

        # Get the top 10 subcategories (if there are more than 10)
        if len(subcategory_counts) > 10:
            print("\nTop 10 Subcategories:")
            top_subcategories = subcategory_counts.head(10)
        else:
            print("\nAll Subcategories:")
            top_subcategories = subcategory_counts

        # Print subcategory distribution
        for i, row in top_subcategories.iterrows():
            print(f"  {row['Category']}: {row['Count']} occurrences")

        # Create detailed subcategory breakdown
        full_subcategory_counts = subcategory_df['full_subcategory'].value_counts().reset_index()
        full_subcategory_counts.columns = ['Subcategory', 'Count']

        # Get the top 15 full subcategories
        if len(full_subcategory_counts) > 15:
            top_full_subcategories = full_subcategory_counts.head(15)
        else:
            top_full_subcategories = full_subcategory_counts

        # Create horizontal bar chart for better readability with long labels
        plt.figure(figsize=(12, max(6, len(top_full_subcategories) * 0.4)))

        # Create horizontal bar chart
        bars = plt.barh(top_full_subcategories['Subcategory'],
                      top_full_subcategories['Count'],
                      color=sns.color_palette("colorblind", len(top_full_subcategories)))

        # Add count labels inside each bar
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width - 0.5, bar.get_y() + bar.get_height()/2.,
                    f'{int(width)}', ha='right', va='center',
                    color='white', fontweight='bold')

        plt.xlabel('Count', fontsize=12)
        plt.ylabel('Subcategory', fontsize=12)
        plt.title(f'Top Error Subcategories{title_suffix}', fontsize=14)
        plt.grid(axis='x', linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.show()

        # Add breakdown by impact if there's enough data
        if 'impact' in subcategory_df.columns and subcategory_df['impact'].nunique() > 1:
            plt.figure(figsize=(12, 6))

            # Count by impact and category
            impact_counts = subcategory_df.groupby(['category', 'impact']).size().reset_index(name='count')

            # Get top categories
            top_categories = subcategory_df['category'].value_counts().head(8).index
            impact_counts = impact_counts[impact_counts['category'].isin(top_categories)]

            # Create grouped bar chart
            sns.barplot(x='category', y='count', hue='impact', data=impact_counts)

            plt.title(f'Error Types by Impact{title_suffix}', fontsize=14)
            plt.xlabel('Error Category', fontsize=12)
            plt.ylabel('Count', fontsize=12)
            plt.legend(title='Impact')
            plt.grid(axis='y', linestyle=':', alpha=0.7)

            # Rotate x-axis labels if needed
            plt.xticks(rotation=45, ha='right')

            plt.tight_layout()
            plt.show()

    return labels_df, subcategory_df if include_subcategories else None

### Error Types Identified - Visualizations

In [49]:
# Basic error type analysis for all documents
visualize_error_types(task_id_to_annotations, include_subcategories=True);

Error Type Analysis (All Documents)
Total annotations analyzed: 2
Total error labels found: 83
Total error subcategories found: 80

Main Error Types Distribution:
  Style: 31 occurrences
  Linguistic Conventions: 20 occurrences
  Accuracy: 18 occurrences
  Terminology: 8 occurrences
  Audience Appropriateness: 6 occurrences

All Subcategories:
  STYLE: 29 occurrences
  LING: 20 occurrences
  ACC: 18 occurrences
  TERM: 8 occurrences
  AUD: 5 occurrences


In [50]:
# Analysis for a specific document with detailed subcategory analysis
visualize_error_types(task_id_to_annotations, doc_id=26, include_subcategories=True);

Error Type Analysis (Document ID: 26)
Total annotations analyzed: 0
Total error labels found: 0
Total error subcategories found: 0
No error labels found in the selected document(s).


### Overall correspondence and readability scores - Code

Here, visualizations on the overall correspondence and readability ratings given to a translation are generated.

In [51]:
def visualize_rating_distributions(processed_data, doc_id=None, plot_type='all'):
    """
    Visualize the distribution of correspondence and readability ratings.

    Args:
        processed_data: Output from preprocess_document_level_data
        doc_id: Optional specific document ID to visualize (None for all documents)
        plot_type: Type of plot to generate ('count', 'box', 'grouped', 'all')

    Returns:
        matplotlib figure(s)
    """
    from matplotlib.gridspec import GridSpec

    # Create a copy of the data to avoid modifying the original
    data = processed_data.copy()

    # Filter by document ID if specified
    if doc_id is not None:
        data = [item for item in data if item.get('task_id') == doc_id]
        title_suffix = f" (Document ID: {doc_id})"
    else:
        title_suffix = " (All Documents)"

    # Extract correspondence and readability ratings
    correspondence_ratings = [item.get('overall_correspondence') for item in data
                            if item.get('overall_correspondence') is not None]
    readability_ratings = [item.get('overall_readability') for item in data
                        if item.get('overall_readability') is not None]

    # Count the occurrences of each rating
    corr_counts = pd.Series(correspondence_ratings).value_counts().sort_index()
    read_counts = pd.Series(readability_ratings).value_counts().sort_index()

    # Prepare a DataFrame for seaborn
    ratings_df = pd.DataFrame({
        'Rating Value': list(range(1, 5)) * 2,
        'Count': [corr_counts.get(i, 0) for i in range(1, 5)] + [read_counts.get(i, 0) for i in range(1, 5)],
        'Category': ['Correspondence'] * 4 + ['Readability'] * 4
    })

    # Long-format for box plots
    long_df = pd.DataFrame({
        'Rating Value': correspondence_ratings + readability_ratings,
        'Category': ['Correspondence'] * len(correspondence_ratings) + ['Readability'] * len(readability_ratings)
    })

    # Print summary statistics
    print(f"Correspondence Ratings{title_suffix}:")
    print(f"  Number of ratings: {len(correspondence_ratings)}")
    print(f"  Mean: {np.mean(correspondence_ratings):.2f}")
    print(f"  Median: {np.median(correspondence_ratings):.1f}")
    print(f"  Mode: {pd.Series(correspondence_ratings).mode().values}")
    print(f"  Distribution: {corr_counts.to_dict()}")

    print(f"\nReadability Ratings{title_suffix}:")
    print(f"  Number of ratings: {len(readability_ratings)}")
    print(f"  Mean: {np.mean(readability_ratings):.2f}")
    print(f"  Median: {np.median(readability_ratings):.1f}")
    print(f"  Mode: {pd.Series(readability_ratings).mode().values}")
    print(f"  Distribution: {read_counts.to_dict()}")

    # Convert to percentages for easier comparison
    ratings_pct = ratings_df.copy()
    corr_total = sum(corr_counts)
    read_total = sum(read_counts)

    for i, row in ratings_pct.iterrows():
        if row['Category'] == 'Correspondence':
            ratings_pct.at[i, 'Percentage'] = (row['Count'] / corr_total * 100) if corr_total > 0 else 0
        else:
            ratings_pct.at[i, 'Percentage'] = (row['Count'] / read_total * 100) if read_total > 0 else 0

    # Create visualizations based on the requested plot type
    if plot_type == 'count':
        # Count Plot
        plt.figure(figsize=(12, 6))
        ax = sns.countplot(x='Rating Value', hue='Category', data=long_df)
        plt.title(f'Distribution of Rating Scores{title_suffix}')
        plt.xlabel('Rating (1-4 scale)')
        plt.ylabel('Count')

        # Add count labels on top of bars
        for p in ax.patches:
            height = p.get_height()
            if height > 0:
                ax.text(p.get_x() + p.get_width()/2.,
                        height + 0.1,
                        f'{height:.0f}',
                        ha="center")

        plt.tight_layout()
        plt.show()

    elif plot_type == 'grouped':
        # Grouped Bar Chart
        plt.figure(figsize=(12, 6))

        # Create the grouped bar chart
        ax = sns.barplot(x='Rating Value', y='Percentage', hue='Category', data=ratings_pct)
        plt.title(f'Percentage of Ratings by Category{title_suffix}')
        plt.xlabel('Rating (1-4 scale)')
        plt.ylabel('Percentage (%)')

        # Add percentage labels on top of bars
        for p in ax.patches:
            height = p.get_height()
            if height > 0:
                ax.text(p.get_x() + p.get_width()/2.,
                        height + 1,
                        f'{height:.1f}%',
                        ha="center")

        plt.tight_layout()
        plt.show()

    elif plot_type == 'all':
        # Combined visualization with all plots
        fig = plt.figure(figsize=(18, 10))
        gs = GridSpec(2, 1, figure=fig)

        # Count plot
        ax1 = fig.add_subplot(gs[0])
        sns.countplot(x='Rating Value', hue='Category', data=long_df, ax=ax1)
        ax1.set_title('Distribution of Rating Scores')
        ax1.set_xlabel('Rating (1-4 scale)')
        ax1.set_ylabel('Count')

        # Grouped bar chart
        ax3 = fig.add_subplot(gs[1])
        sns.barplot(x='Rating Value', y='Percentage', hue='Category', data=ratings_pct, ax=ax3)
        ax3.set_title('Percentage of Ratings by Category')
        ax3.set_xlabel('Rating (1-4 scale)')
        ax3.set_ylabel('Percentage (%)')

        fig.suptitle(f'Rating Analysis{title_suffix}', fontsize=16)
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        plt.show()

    return ratings_df, long_df

### Overall correspondence and readability scores - Visualizations

#### All visualizations

In [52]:
# Show all plot types for all documents in the project
visualize_rating_distributions(processed_data, doc_id=None, plot_type='all');

Correspondence Ratings (All Documents):
  Number of ratings: 2
  Mean: 3.00
  Median: 3.0
  Mode: [3]
  Distribution: {3: 2}

Readability Ratings (All Documents):
  Number of ratings: 2
  Mean: 2.00
  Median: 2.0
  Mode: [1 3]
  Distribution: {1: 1, 3: 1}


In [53]:
# Show all plot types for a specific document
visualize_rating_distributions(processed_data, doc_id=26, plot_type='all');

Correspondence Ratings (Document ID: 26):
  Number of ratings: 0
  Mean: nan
  Median: nan
  Mode: []
  Distribution: {}

Readability Ratings (Document ID: 26):
  Number of ratings: 0
  Mean: nan
  Median: nan
  Mode: []
  Distribution: {}


#### Count plot on rating distributions

**How to Interpret Rating Distribution Charts**

This chart visualizes the distribution of translation quality ratings across two assessment categories: correspondence and readability.

*Reading the Count Plot*
- X-axis: Shows the rating scale (1-4), where 1 is lowest quality and 4 is highest quality
- Y-axis: Shows the count (number of annotators) who assigned each rating
- Color: Distinguishes between Readability (blue) and Correspondence (orange) ratings

*Key Insights*
- Rating Distribution: Taller bars indicate more common ratings
- Rating Patterns: Compare blue vs. orange bars to see differences between how annotators rated readability vs. correspondence
- Agreement Level: Bars concentrated at one rating value suggest stronger annotator agreement
- Quality Assessment: The concentration of ratings at higher numbers (3-4) indicates better perceived translation quality

*Document selection*

This visualization aggregates ratings across all documents. To view ratings for a specific document, use the document ID parameter when calling the visualization function.

Show only count plot for a single documents: visualize_rating_distributions(processed_data, doc_id=1, plot_type='count');

In [54]:
# Show only count plot for all documents
visualize_rating_distributions(processed_data, doc_id=None, plot_type='count');

Correspondence Ratings (All Documents):
  Number of ratings: 2
  Mean: 3.00
  Median: 3.0
  Mode: [3]
  Distribution: {3: 2}

Readability Ratings (All Documents):
  Number of ratings: 2
  Mean: 2.00
  Median: 2.0
  Mode: [1 3]
  Distribution: {1: 1, 3: 1}


#### Grouped bar chart of rating distributions

In [55]:
# Show only grouped bar chart for a specific document
visualize_rating_distributions(processed_data, doc_id=26, plot_type='grouped');

Correspondence Ratings (Document ID: 26):
  Number of ratings: 0
  Mean: nan
  Median: nan
  Mode: []
  Distribution: {}

Readability Ratings (Document ID: 26):
  Number of ratings: 0
  Mean: nan
  Median: nan
  Mode: []
  Distribution: {}


## 5. Translation Error Annotations - Agreement Calculations

- Exact matching for span boundaries
- F1 for partial credit on spans
- Kappa for category agreement, subcategory agreement, and impact levels

### 5.1 Exact Matching for Span Boundaries


**How it works**
- Measures when two or more annotators identify exactly the same text span (identical start and end positions)
- Typically expressed as a percentage: "Annotators agreed on exact span boundaries for 65% of errors"

**What you'll learn**
- How precisely annotators agree on where errors begin and end
- Identifies potential ambiguity in annotation guidelines
- Shows if annotators are consistently identifying the same textual issues

In MQM translation error annotation context, exact matching shows how often annotators precisely agree on which specific words or phrases contain errors.

#### **Exact Matching for Span Boundaries - Code**

In [56]:
def calculate_exact_matches(task_id_to_annotations):
    """
    Calculate exact matches between annotators for each document.

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations

    Returns:
        Dictionary with exact match statistics
    """
    # Results structure
    results = {
        'per_document': {},
        'overall': {
            'total_comparisons': 0,
            'total_exact_matches': 0,
            'exact_match_rate': 0
        }
    }

    # Process each document
    for doc_id, annotations in task_id_to_annotations.items():
        # Group annotations by annotator
        annotator_to_spans = {}

        for annotation in annotations:
            annotator = annotation.get('annotator_id')
            if not annotator:
                continue

            # Get all span boundaries for this annotator
            spans = []
            for label in annotation.get('enriched_labels', []):
                spans.append({
                    'start': label.get('start'),
                    'end': label.get('end'),
                    'text': label.get('text'),
                    'category': label.get('labels', [''])[0] if label.get('labels') else '',
                    'subcategory': label.get('subcategory', ''),
                    'impact': label.get('impact', '')
                })

            annotator_to_spans[annotator] = spans

        # Skip documents with fewer than 2 annotators
        if len(annotator_to_spans) < 2:
            continue

        # Initialize document results
        doc_results = {
            'annotator_pairs': [],
            'total_comparisons': 0,
            'total_exact_matches': 0,
            'exact_match_rate': 0
        }

        # Compare each pair of annotators
        for annotator1, annotator2 in combinations(annotator_to_spans.keys(), 2):
            spans1 = annotator_to_spans[annotator1]
            spans2 = annotator_to_spans[annotator2]

            # Count exact matches
            exact_matches = 0

            # Create sets of (start, end) tuples for each annotator
            spans1_set = {(span['start'], span['end']) for span in spans1}
            spans2_set = {(span['start'], span['end']) for span in spans2}

            # Find intersection (exact matches)
            exact_match_set = spans1_set.intersection(spans2_set)
            exact_matches = len(exact_match_set)

            # Calculate comparison stats
            total_spans = len(spans1) + len(spans2)
            unique_spans = len(spans1_set.union(spans2_set))

            # Avoid division by zero
            exact_match_rate = exact_matches / unique_spans if unique_spans > 0 else 0

            # Store pair results
            pair_result = {
                'annotator1': annotator1,
                'annotator2': annotator2,
                'spans1': len(spans1),
                'spans2': len(spans2),
                'unique_spans': unique_spans,
                'exact_matches': exact_matches,
                'exact_match_rate': exact_match_rate
            }

            doc_results['annotator_pairs'].append(pair_result)
            doc_results['total_comparisons'] += unique_spans
            doc_results['total_exact_matches'] += exact_matches

        # Calculate overall document exact match rate
        if doc_results['total_comparisons'] > 0:
            doc_results['exact_match_rate'] = doc_results['total_exact_matches'] / doc_results['total_comparisons']

        # Store document results
        results['per_document'][doc_id] = doc_results

        # Update overall statistics
        results['overall']['total_comparisons'] += doc_results['total_comparisons']
        results['overall']['total_exact_matches'] += doc_results['total_exact_matches']

    # Calculate overall exact match rate
    if results['overall']['total_comparisons'] > 0:
        results['overall']['exact_match_rate'] = results['overall']['total_exact_matches'] / results['overall']['total_comparisons']

    return results

In [57]:
def display_exact_match_results(exact_match_results):
    """
    Display exact match results in a readable format with visualizations.

    Args:
        exact_match_results: Output from calculate_exact_matches function
    """
    print("=" * 60)
    print("EXACT MATCH ANALYSIS FOR SPAN BOUNDARIES")
    print("=" * 60)

    # Overall results
    overall = exact_match_results['overall']
    print(f"\nOVERALL RESULTS:")
    print(f"Total comparisons: {overall['total_comparisons']}")
    print(f"Total exact matches: {overall['total_exact_matches']}")
    print(f"Exact match rate: {overall['exact_match_rate']:.2%}")

    # Per-document results
    print("\nRESULTS BY DOCUMENT:")
    for doc_id, doc_results in exact_match_results['per_document'].items():
        print(f"\nDocument {doc_id}:")
        print(f"  Exact match rate: {doc_results['exact_match_rate']:.2%}")
        print(f"  Exact matches: {doc_results['total_exact_matches']} out of {doc_results['total_comparisons']}")

        # Results by annotator pair
        print("\n  Results by annotator pair:")
        for pair in doc_results['annotator_pairs']:
            print(f"    Annotators {pair['annotator1']} and {pair['annotator2']}:")
            print(f"      Spans: {pair['spans1']} and {pair['spans2']}")
            print(f"      Exact matches: {pair['exact_matches']} out of {pair['unique_spans']} unique spans")
            print(f"      Exact match rate: {pair['exact_match_rate']:.2%}")

    # Create visualization
    plt.figure(figsize=(10, 6))

    # Prepare data for bar chart
    doc_ids = []
    doc_rates = []

    for doc_id, doc_results in exact_match_results['per_document'].items():
        doc_ids.append(f"Doc {doc_id}")
        doc_rates.append(doc_results['exact_match_rate'] * 100)

    # Add overall rate
    # Plot
    bars = plt.bar(doc_ids, doc_rates, color='skyblue')

    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom')

    plt.title('Exact Match Rate by Document')
    plt.xlabel('Document')
    plt.ylabel('Exact Match Rate (%)')
    plt.ylim(0, max(max(doc_rates) * 1.2, 10))
    plt.tight_layout()

    plt.show()

In [58]:
# Calculate exact matches
exact_match_results = calculate_exact_matches(task_id_to_annotations)

#### **Exact Matching for Span Boundaries - Visualizations**

In [59]:
# Display results
display_exact_match_results(exact_match_results)

EXACT MATCH ANALYSIS FOR SPAN BOUNDARIES

OVERALL RESULTS:
Total comparisons: 74
Total exact matches: 9
Exact match rate: 12.16%

RESULTS BY DOCUMENT:

Document 32:
  Exact match rate: 12.16%
  Exact matches: 9 out of 74

  Results by annotator pair:
    Annotators 58 and 61:
      Spans: 49 and 34
      Exact matches: 9 out of 74 unique spans
      Exact match rate: 12.16%


### 5.2 F1 for Partial Credit on Spans

**How F1 Calculation Works**

The core concept: Instead of requiring exact span boundaries to match, we:
- Give credit when spans overlap
- Calculate how much they overlap relative to each annotator's span (precision and recall)
- Combine these into an F1 score
- Values range from 0 (no overlap) to 1 (perfect overlap)

For each span:
- Precision: What percentage of annotator 1's span is overlapped by annotator 2's span
- Recall: What percentage of annotator 2's span is overlapped by annotator 1's span
- F1: The harmonic mean of precision and recall (balances both)

Matching process:
- For each span from annotator 1, find the span from annotator 2 that gives the highest F1 score
- If multiple spans from annotator 2 overlap with a span from annotator 1, only the best match counts

Aggregation:
- Calculate average precision, recall, and F1 for each annotator pair
- Calculate averages per document and overall across all documents

**What You'll Learn From F1 Scores**

F1 scores give a more nuanced view of agreement than exact matching. For example, if one annotator marks "aerial carbon" as an error while another marks "aerial carbon in trees", F1 would give partial credit for this overlap.

F1 scores are likely be much higher than exact match percentages, showing that:
- Annotators are often identifying the same errors but choosing slightly different span boundaries
- Some annotator pairs have more similar annotation styles than others
- Some documents may have more clearly defined errors, leading to higher agreement

The F1 visualization will help you identify which annotators tend to agree more with others, and which documents have the most consistent annotations.

#### **F1 for Partial Credit on Spans - Code**

In [60]:
def calculate_span_f1(task_id_to_annotations, overlap_threshold=0.0):
    """
    Calculate F1 scores for partial span matches between annotators.

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations
        overlap_threshold: Minimum overlap ratio required to consider spans as matching (0.0 means any overlap)

    Returns:
        Dictionary with F1 statistics
    """
    # Results structure
    results = {
        'per_document': {},
        'overall': {
            'total_comparisons': 0,
            'avg_precision': 0,
            'avg_recall': 0,
            'avg_f1': 0
        }
    }

    overall_precisions = []
    overall_recalls = []
    overall_f1s = []

    # Process each document
    for doc_id, annotations in task_id_to_annotations.items():
        # Group annotations by annotator
        annotator_to_spans = {}

        for annotation in annotations:
            annotator = annotation.get('annotator_id')
            if not annotator:
                continue

            # Get all span boundaries for this annotator
            spans = []
            for label in annotation.get('enriched_labels', []):
                spans.append({
                    'start': label.get('start'),
                    'end': label.get('end'),
                    'text': label.get('text'),
                    'length': label.get('end') - label.get('start'),
                    'category': label.get('labels', [''])[0] if label.get('labels') else '',
                    'subcategory': label.get('subcategory', ''),
                    'impact': label.get('impact', '')
                })

            annotator_to_spans[annotator] = spans

        # Skip documents with fewer than 2 annotators
        if len(annotator_to_spans) < 2:
            continue

        # Initialize document results
        doc_results = {
            'annotator_pairs': [],
            'avg_precision': 0,
            'avg_recall': 0,
            'avg_f1': 0
        }

        doc_precisions = []
        doc_recalls = []
        doc_f1s = []

        # Compare each pair of annotators
        for annotator1, annotator2 in combinations(annotator_to_spans.keys(), 2):
            spans1 = annotator_to_spans[annotator1]
            spans2 = annotator_to_spans[annotator2]

            # Skip if either annotator has no spans
            if not spans1 or not spans2:
                continue

            # Create binary arrays representing each character position
            # This is a more rigorous way to calculate true precision/recall
            max_pos = max([span['end'] for spans in [spans1, spans2] for span in spans])
            array1 = np.zeros(max_pos + 1, dtype=bool)
            array2 = np.zeros(max_pos + 1, dtype=bool)

            # Mark positions covered by each annotator's spans
            for span in spans1:
                array1[span['start']:span['end']] = True
            for span in spans2:
                array2[span['start']:span['end']] = True

            # Calculate true precision and recall using the character arrays
            # Precision: proportion of characters marked by annotator1 that are also marked by annotator2
            # Recall: proportion of characters marked by annotator2 that are also marked by annotator1
            true_positives = np.sum(array1 & array2)
            total_predicted = np.sum(array1)
            total_actual = np.sum(array2)

            precision = true_positives / total_predicted if total_predicted > 0 else 0
            recall = true_positives / total_actual if total_actual > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # Track match statistics for reporting purposes
            # Find overlapping spans for detailed reporting
            matches = []
            for span1 in spans1:
                for span2 in spans2:
                    overlap_start = max(span1['start'], span2['start'])
                    overlap_end = min(span1['end'], span2['end'])

                    if overlap_start < overlap_end:  # Spans overlap
                        matches.append({
                            'span1': span1,
                            'span2': span2,
                            'overlap_length': overlap_end - overlap_start
                        })

            # Store pair results
            pair_result = {
                'annotator1': annotator1,
                'annotator2': annotator2,
                'spans1': len(spans1),
                'spans2': len(spans2),
                'overlap_matches': len(matches),
                'precision': precision,
                'recall': recall,
                'f1': f1
            }

            doc_precisions.append(precision)
            doc_recalls.append(recall)
            doc_f1s.append(f1)

            overall_precisions.append(precision)
            overall_recalls.append(recall)
            overall_f1s.append(f1)

            doc_results['annotator_pairs'].append(pair_result)

        # Calculate document averages
        if doc_precisions:
            doc_results['avg_precision'] = sum(doc_precisions) / len(doc_precisions)
            doc_results['avg_recall'] = sum(doc_recalls) / len(doc_recalls)
            doc_results['avg_f1'] = sum(doc_f1s) / len(doc_f1s)

        # Store document results
        results['per_document'][doc_id] = doc_results

    # Calculate overall averages
    if overall_precisions:
        results['overall']['avg_precision'] = sum(overall_precisions) / len(overall_precisions)
        results['overall']['avg_recall'] = sum(overall_recalls) / len(overall_recalls)
        results['overall']['avg_f1'] = sum(overall_f1s) / len(overall_f1s)
        results['overall']['total_comparisons'] = len(overall_f1s)

    return results

In [61]:
def display_span_f1_results(f1_results):
    """
    Display F1 results in a readable format with visualizations.

    Args:
        f1_results: Output from calculate_span_f1 function
    """
    print("=" * 60)
    print("F1 ANALYSIS FOR PARTIAL SPAN MATCHES")
    print("=" * 60)

    # Overall results
    overall = f1_results['overall']
    print(f"\nOVERALL RESULTS:")
    print(f"Total annotator pair comparisons: {overall['total_comparisons']}")
    print(f"Average Precision: {overall['avg_precision']:.2%}")
    print(f"Average Recall: {overall['avg_recall']:.2%}")
    print(f"Average F1 Score: {overall['avg_f1']:.2%}")

    # Per-document results
    print("\nRESULTS BY DOCUMENT:")
    for doc_id, doc_results in f1_results['per_document'].items():
        print(f"\nDocument {doc_id}:")
        print(f"  Average F1 Score: {doc_results['avg_f1']:.2%}")
        print(f"  Average Precision: {doc_results['avg_precision']:.2%}")
        print(f"  Average Recall: {doc_results['avg_recall']:.2%}")

        # Results by annotator pair
        print("\n  Results by annotator pair:")
        for pair in sorted(doc_results['annotator_pairs'], key=lambda x: x['f1'], reverse=True):
            print(f"    Annotators {pair['annotator1']} and {pair['annotator2']}:")
            print(f"      Spans: {pair['spans1']} and {pair['spans2']}")
            print(f"      Overlapping spans: {pair['overlap_matches']}")  # Changed from 'matches' to 'overlap_matches'
            print(f"      Precision: {pair['precision']:.2%}")
            print(f"      Recall: {pair['recall']:.2%}")
            print(f"      F1 Score: {pair['f1']:.2%}")

    # Create visualization
    plt.figure(figsize=(12, 8))

    # Prepare data for bar chart (single document — no Overall column)
    doc_ids = []
    doc_f1s = []
    doc_precisions = []
    doc_recalls = []
    doc_exact_rates = []

    for doc_id, doc_results in f1_results['per_document'].items():
        doc_ids.append(f"Doc {doc_id}")
        doc_f1s.append(doc_results['avg_f1'] * 100)
        doc_precisions.append(doc_results['avg_precision'] * 100)
        doc_recalls.append(doc_results['avg_recall'] * 100)
        # Pull exact match rate from exact_match_results if available in scope
        try:
            doc_exact_rates.append(exact_match_results['per_document'][doc_id]['exact_match_rate'] * 100)
        except (NameError, KeyError):
            doc_exact_rates.append(0)

    # Set bar width
    barWidth = 0.2

    # Set position of bars on X axis
    r1 = np.arange(len(doc_ids))
    r2 = [x + barWidth for x in r1]
    r3 = [x + barWidth for x in r2]
    r4 = [x + barWidth for x in r3]

    # Create grouped bars: Exact Match F1, F1, Precision, Recall
    plt.bar(r1, doc_exact_rates, width=barWidth, color='#ff66ff', label='Exact Match Rate')
    plt.bar(r2, doc_f1s, width=barWidth, color='blue', label='F1 Score')
    plt.bar(r3, doc_precisions, width=barWidth, color='green', label='Precision')
    plt.bar(r4, doc_recalls, width=barWidth, color='orange', label='Recall')

    # Add value labels
    for bars_group in [
        (r1, doc_exact_rates),
        (r2, doc_f1s),
        (r3, doc_precisions),
        (r4, doc_recalls),
    ]:
        for rx, val in zip(bars_group[0], bars_group[1]):
            plt.text(rx, val + 1, f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

    # Add labels and legend
    plt.xlabel('Document')
    plt.ylabel('Score (%)')
    plt.title('Exact Match Rate, F1, Precision, and Recall')
    plt.xticks([r + barWidth * 1.5 for r in range(len(doc_ids))], doc_ids)
    plt.ylim(0, 110)
    plt.legend()

    plt.tight_layout()
    plt.show()

In [62]:
# Calculate F1 scores for partial matches
# Any overlap is considered a match (threshold=0.0)
f1_results = calculate_span_f1(task_id_to_annotations, overlap_threshold=0.0)

#### **F1 for Partial Credit on Spans - Visualizations**

In [63]:
# Display results
display_span_f1_results(f1_results)

F1 ANALYSIS FOR PARTIAL SPAN MATCHES

OVERALL RESULTS:
Total annotator pair comparisons: 1
Average Precision: 49.20%
Average Recall: 65.85%
Average F1 Score: 56.32%

RESULTS BY DOCUMENT:

Document 32:
  Average F1 Score: 56.32%
  Average Precision: 49.20%
  Average Recall: 65.85%

  Results by annotator pair:
    Annotators 58 and 61:
      Spans: 49 and 34
      Overlapping spans: 25
      Precision: 49.20%
      Recall: 65.85%
      F1 Score: 56.32%


## 5.3 Kappa for Category Agreement

**How it works**
- Cohen's Kappa (two annotators) or Fleiss' Kappa (multiple annotators) measures agreement on categories while accounting for chance agreement
- Scale typically ranges from -1 to 1:
 - < 0.20: Poor agreement
 - 0.21-0.40: Fair agreement
 - 0.41-0.60: Moderate agreement
 - 0.61-0.80: Substantial agreement
 - 0.81-1.00: Almost perfect agreement

**Notes**

Cohen's κ measures agreement on error category labels for overlapping spans. Production deployment target for both metrics: ≥0.60. Low reliability in one condition means its quality metrics should be interpreted with caution.

**What you could potentially learn**
- How consistently annotators classify errors into categories (Accuracy, Terminology, Style)
- How consistently annotators use subcategories (e.g., "TERM: Wrong term")
- How consistently annotators assign impact levels (Neutral/Moderate/Strong/Showstopper)
- Whether certain categories have better agreement than others

This is particularly valuable for the MQM framework since it has a complex hierarchy of error types.

**Challenges inherent in this approach**
- Low exact match rates and F1 scores present serious challenges for calculating meaningful Kappa agreement
- There will be even lower agreement as we transition from each level of agreement to the next:
 - Span agreement
 - Label agreement
 - Subcategory agreement
 - Impact agreement

**Parameters implemented to respond to challenges**
- Finding spans with substantial overlap (≥50%)
- Only calculating Kappa for these overlapping spans

**Recommendations**

Where there are low exact match rates, F1 scores, and Kappa scores for category agreement, proceeding to Kappa scoring for subcategory agreement and impact agreement is not recommended.

#### **Cohen's Kappa for Category Agreement - Code**

In [64]:
def calculate_category_agreement(task_id_to_annotations, attribute='labels'):
    """
    Calculate Cohen's Kappa for agreement on categories, subcategories, or impact levels
    across annotator pairs.

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations
        attribute: The attribute to measure agreement on ('labels', 'subcategory', or 'impact')

    Returns:
        Dictionary with agreement statistics
    """
    # Results structure
    results = {
        'per_document': {},
        'overall': {
            'avg_kappa': 0,
            'category_counts': {},
            'pair_count': 0
        }
    }

    # Overall statistics
    all_kappas = []
    all_weights = []
    all_category_counts = {}

    # Process each document
    for doc_id, annotations in task_id_to_annotations.items():
        # Group annotations by annotator
        annotator_to_spans = {}

        for annotation in annotations:
            annotator = annotation.get('annotator_id')
            if not annotator:
                continue

            # Extract spans for this annotator
            spans = []
            for label in annotation.get('enriched_labels', []):
                span = {
                    'start': label.get('start'),
                    'end': label.get('end'),
                    'text': label.get('text')
                }

                # Extract the relevant attribute value
                if attribute == 'labels':
                    # For labels, take the first item from the list
                    span['category'] = label.get('labels', [''])[0] if label.get('labels') else ''
                elif attribute == 'subcategory':
                    span['category'] = label.get('subcategory', '')
                elif attribute == 'impact':
                    span['category'] = label.get('impact', '')
                else:
                    continue

                spans.append(span)

            # Store spans for this annotator
            annotator_to_spans[annotator] = spans

        # Skip documents with fewer than 2 annotators
        if len(annotator_to_spans) < 2:
            continue

        # Initialize document results
        doc_results = {
            'annotator_pairs': [],
            'avg_kappa': 0,
            'category_counts': {},
            'pair_count': 0
        }

        # Calculate agreement for each pair of annotators
        doc_kappas = []
        doc_weights = []
        doc_category_counts = {}

        for annotator1, annotator2 in combinations(annotator_to_spans.keys(), 2):
            spans1 = annotator_to_spans[annotator1]
            spans2 = annotator_to_spans[annotator2]

            # Find overlapping spans
            overlapping_spans = []

            for span1 in spans1:
                for span2 in spans2:
                    # Check for overlap
                    if span1['start'] < span2['end'] and span2['start'] < span1['end']:
                        # Calculate overlap
                        overlap_start = max(span1['start'], span2['start'])
                        overlap_end = min(span1['end'], span2['end'])
                        overlap_len = overlap_end - overlap_start

                        # Calculate overlap ratio relative to the smaller span
                        span1_len = span1['end'] - span1['start']
                        span2_len = span2['end'] - span2['start']
                        min_len = min(span1_len, span2_len)
                        overlap_ratio = overlap_len / min_len if min_len > 0 else 0

                        # Only consider substantial overlaps (at least 50%)
                        if overlap_ratio >= 0.5:
                            overlapping_spans.append({
                                'span1': span1,
                                'span2': span2,
                                'overlap_ratio': overlap_ratio
                            })

            # Skip pairs with too few overlapping spans for Kappa
            if len(overlapping_spans) < 2:
                continue

            # Extract categories for overlapping spans
            categories1 = [overlap['span1']['category'] for overlap in overlapping_spans]
            categories2 = [overlap['span2']['category'] for overlap in overlapping_spans]

            # Count category frequencies
            for category in categories1 + categories2:
                doc_category_counts[category] = doc_category_counts.get(category, 0) + 1
                all_category_counts[category] = all_category_counts.get(category, 0) + 1

            # Calculate Cohen's Kappa
            try:
                kappa = cohen_kappa_score(categories1, categories2)
                weight = len(overlapping_spans)

                doc_kappas.append(kappa)
                doc_weights.append(weight)

                all_kappas.append(kappa)
                all_weights.append(weight)

                # Record pair results
                pair_result = {
                    'annotator1': annotator1,
                    'annotator2': annotator2,
                    'overlapping_spans': len(overlapping_spans),
                    'kappa': kappa
                }

                doc_results['annotator_pairs'].append(pair_result)
                doc_results['pair_count'] += 1
            except Exception as e:
                # Skip pairs with invalid Kappa calculation
                print(f"Warning: Could not calculate Kappa for annotators {annotator1} and {annotator2} - {e}")
                continue

        # Calculate weighted average Kappa for this document
        if doc_kappas:
            total_weight = sum(doc_weights)
            doc_results['avg_kappa'] = sum(k * w for k, w in zip(doc_kappas, doc_weights)) / total_weight
            doc_results['category_counts'] = doc_category_counts

        # Store document results
        results['per_document'][doc_id] = doc_results

    # Calculate overall weighted average Kappa
    if all_kappas:
        total_weight = sum(all_weights)
        results['overall']['avg_kappa'] = sum(k * w for k, w in zip(all_kappas, all_weights)) / total_weight
        results['overall']['category_counts'] = all_category_counts
        results['overall']['pair_count'] = len(all_kappas)

    return results

In [65]:
def display_category_agreement(agreement_results, attribute_name):
    """
    Display category agreement results in a readable format with visualizations.

    Args:
        agreement_results: Output from calculate_category_agreement function
        attribute_name: Name of the attribute (e.g., 'Category', 'Subcategory', 'Impact')
    """
    print("=" * 60)
    print(f"COHEN'S KAPPA ANALYSIS FOR {attribute_name.upper()} AGREEMENT")
    print("=" * 60)

    # Overall results
    overall = agreement_results['overall']
    print(f"\nOVERALL RESULTS:")

    # Interpret Kappa value
    kappa = overall['avg_kappa']
    interpretation = "Poor agreement"
    if kappa > 0.80:
        interpretation = "Almost perfect agreement"
    elif kappa > 0.60:
        interpretation = "Substantial agreement"
    elif kappa > 0.40:
        interpretation = "Moderate agreement"
    elif kappa > 0.20:
        interpretation = "Fair agreement"

    print(f"Average Cohen's Kappa: {kappa:.4f} - {interpretation}")
    print(f"Total annotator pairs analyzed: {overall['pair_count']}")

    # Category-specific results
    print(f"\n{attribute_name.upper()} FREQUENCY:")
    total_count = sum(overall['category_counts'].values())

    # Sort categories by frequency
    sorted_categories = sorted(overall['category_counts'].items(), key=lambda x: x[1], reverse=True)

    for category, count in sorted_categories:
        frequency = count / total_count if total_count > 0 else 0
        print(f"  {category}: {frequency:.2%}")

    # Per-document results
    print("\nRESULTS BY DOCUMENT:")
    for doc_id, doc_results in agreement_results['per_document'].items():
        doc_kappa = doc_results['avg_kappa']
        doc_interpretation = "Poor agreement"
        if doc_kappa > 0.80:
            doc_interpretation = "Almost perfect agreement"
        elif doc_kappa > 0.60:
            doc_interpretation = "Substantial agreement"
        elif doc_kappa > 0.40:
            doc_interpretation = "Moderate agreement"
        elif doc_kappa > 0.20:
            doc_interpretation = "Fair agreement"

        print(f"\nDocument {doc_id}:")
        print(f"  Kappa: {doc_kappa:.4f} - {doc_interpretation}")
        print(f"  Annotator pairs: {doc_results['pair_count']}")

        # Display results by annotator pair
        if doc_results['annotator_pairs']:
            print("\n  Results by annotator pair:")
            for pair in sorted(doc_results['annotator_pairs'], key=lambda x: x['kappa'], reverse=True):
                print(f"    Annotators {pair['annotator1']} and {pair['annotator2']}:")
                print(f"      Overlapping spans: {pair['overlapping_spans']}")
                print(f"      Kappa: {pair['kappa']:.4f}")

    # Create visualization
    plt.figure(figsize=(12, 8))

    # Prepare data for bar chart
    doc_ids = [f"Doc {doc_id}" for doc_id in agreement_results['per_document'].keys()]
    doc_kappas = [doc_result['avg_kappa'] for doc_result in agreement_results['per_document'].values()]

    # Add overall kappa
    doc_ids.append("Overall")
    doc_kappas.append(overall['avg_kappa'])

    # Color bars based on Kappa interpretation
    colors = []
    for kappa in doc_kappas:
        if kappa > 0.80:
            colors.append('darkgreen')
        elif kappa > 0.60:
            colors.append('green')
        elif kappa > 0.40:
            colors.append('yellow')
        elif kappa > 0.20:
            colors.append('orange')
        else:
            colors.append('red')

    # Create bar chart
    bars = plt.bar(doc_ids, doc_kappas, color=colors)

    # Add horizontal lines for Kappa interpretation thresholds
    plt.axhline(y=0.20, color='r', linestyle='--', alpha=0.5, label='Poor/Fair')
    plt.axhline(y=0.40, color='orange', linestyle='--', alpha=0.5, label='Fair/Moderate')
    plt.axhline(y=0.60, color='y', linestyle='--', alpha=0.5, label='Moderate/Substantial')
    plt.axhline(y=0.80, color='g', linestyle='--', alpha=0.5, label='Substantial/Almost Perfect')

    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom')

    # Add labels and title
    plt.xlabel('Document')
    plt.ylabel("Cohen's Kappa")
    plt.title(f"Average Cohen's Kappa for {attribute_name} Agreement by Document")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.tight_layout()

    # Create a second chart for category frequencies
    if sorted_categories:
        plt.figure(figsize=(14, 6))

        # Prepare data (show top 10 categories)
        top_n = min(10, len(sorted_categories))
        top_categories = [cat for cat, _ in sorted_categories[:top_n]]
        frequencies = [count/total_count for cat, count in sorted_categories[:top_n]]

        # Create horizontal bar chart
        bars = plt.barh(top_categories, frequencies, color='skyblue')

        # Add value labels
        for bar in bars:
            width = bar.get_width()
            plt.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{width:.1%}', va='center')

        # Add labels and title
        plt.xlabel('Frequency')
        plt.ylabel(attribute_name)
        plt.title(f"{attribute_name} Frequency Distribution")
        plt.xlim(0, max(frequencies) * 1.1 if frequencies else 0.1)
        plt.tight_layout()

    plt.show()

In [66]:
def analyze_category_agreement(task_id_to_annotations):
    """
    Calculate and display agreement metrics for main error categories.

    Args:
        task_id_to_annotations: Dictionary mapping doc_ids to lists of annotations

    Returns:
        The calculated agreement metrics
    """
    print("\nCalculating agreement for main categories...")
    category_agreement = calculate_category_agreement(task_id_to_annotations, attribute='labels')
    display_category_agreement(category_agreement, attribute_name='Category')

    return category_agreement

#### **Cohen's Kappa for Category Agreement - Visualization**

In [67]:
# Calculate and display Kappa for error categories
category_agreement = analyze_category_agreement(task_id_to_annotations)


Calculating agreement for main categories...
COHEN'S KAPPA ANALYSIS FOR CATEGORY AGREEMENT

OVERALL RESULTS:
Average Cohen's Kappa: 0.7859 - Substantial agreement
Total annotator pairs analyzed: 1

CATEGORY FREQUENCY:
  Style: 36.00%
  Accuracy: 26.00%
  Linguistic Conventions: 20.00%
  Audience Appropriateness: 10.00%
  Terminology: 8.00%

RESULTS BY DOCUMENT:

Document 32:
  Kappa: 0.7859 - Substantial agreement
  Annotator pairs: 1

  Results by annotator pair:
    Annotators 58 and 61:
      Overlapping spans: 25
      Kappa: 0.7859


#### **Why Fleiss's Kappa Is Not Implemented**

Fleiss's Kappa extends Cohen's Kappa to handle more than two annotators simultaneously, producing a single agreement estimate across all raters rather than averaging pairwise scores.

**What implementation would require**

Unlike Cohen's Kappa, Fleiss's requires a fixed rating matrix where every rater judges every item. Span annotations don't naturally produce this — each annotator identifies their own spans independently. The standard workaround is to divide the source text into fixed-width character windows (e.g. every 25 characters = one item) and record whether each annotator flagged anything in that window (1) or not (0).

**Challenges with this approach**

- **Window size requires experimentation.** Too small and a single span artificially triggers many consecutive windows; too large and you lose meaningful granularity. There is no universally correct value.
- **Spans crossing window boundaries.** A span covering characters 20-30 split across a 0–25 / 26–50 boundary would need a decision rule — count it in both windows, the window where it starts, or the window where most of it falls. Each choice affects the result.
- **Loss of category information.** The windowed approach only captures whether something was flagged, not what it was classified as — making it a measure of span location agreement only, not category agreement.

**Why we are not implementing it yet**

Fleiss's Kappa would be most useful as a single summary statistic once annotators are reaching consistent span and category agreement. Given current Cohen's Kappa and F1 scores, the underlying agreement is not yet at a level where Fleiss would add meaningful signal over the pairwise results. The pairwise Cohen's scores are also more diagnostic, making it easier to identify individual annotators whose labelling behaviour diverges from the group. We recommend revisiting Fleiss's Kappa once Cohen's Kappa scores are consistently in the moderate range (≥ 0.40) across annotator pairs.

## 6. Export Report

Run this cell to produce a self-contained HTML report for sharing with evaluators.

**Configuration options:**
- Toggle each section `True`/`False` to include or exclude it from the report.
- `error_type_distribution_doc` — set to a document ID (e.g. `24`) to show error charts for one document only, or `None` for all documents.
- `annotator_completeness_doc` — same scoping option for the completeness tables.
- `cohens_kappa_categories` replaces the previous `kappa_category` key — the name now explicitly identifies this as a Cohen's Kappa analysis (pairwise, not Fleiss').

The report file is written to `OUTPUT_DIR` with the project name and date in the filename.

**Tip:** turn off sections that aren't meaningful yet — for example, if only one annotator has submitted, agreement metrics will be empty.


In [68]:
# ── EXPORT CONFIG ─────────────────────────────────────────────────────────────
# Toggle each section True/False.
# For each section, the paired _doc key controls scope:
#   None  → whole-project report (all documents)
#   24    → single-document report (replace 24 with a valid document ID)

EXPORT = {
    # Whole-project reports: set _doc to None
    # Single-document reports: set _doc to an int, e.g. 24

    "annotator_completeness":          True,
    "annotator_completeness_doc":      None,

    "annotator_timing":                True,
    "annotator_timing_doc":            None,

    "span_overlap_visualization":      True,
    "span_overlap_visualization_doc":  None,

    "error_type_distribution":         True,
    "error_type_distribution_doc":     None,

    "rating_distributions":            True,
    "rating_distributions_doc":        None,

    "exact_span_matching":             True,
    "exact_span_matching_doc":         None,

    "span_f1":                         True,
    "span_f1_doc":                     None,

    "cohens_kappa_categories":         True,
    "cohens_kappa_categories_doc":     None,
}
# ──────────────────────────────────────────────────────────────────────────────

import os
from IPython.display import display, HTML

os.makedirs(OUTPUT_DIR, exist_ok=True)
def _unique_path(directory, filename):
    """Return a path that doesn't overwrite an existing file, appending (1), (2) etc."""
    base, ext = os.path.splitext(filename)
    candidate = os.path.join(directory, filename)
    counter = 1
    while os.path.exists(candidate):
        candidate = os.path.join(directory, f"{base} ({counter}){ext}")
        counter += 1
    return candidate

report_filename = f"{PROJECT_NAME}_report_{REPORT_DATE}.html"
report_path     = _unique_path(OUTPUT_DIR, report_filename)

sections_html = []

def section(title, content_html, description=None):
    desc_html = f'<p class="section-desc">{description}</p>' if description else ""
    return f"""
    <section style="margin-bottom: 2.5rem;">
      <h2 style="border-bottom: 2px solid #ccc; padding-bottom: .4rem;">{title}</h2>
      {desc_html}
      {content_html}
    </section>"""

def fig_to_html(fig):
    """Convert a matplotlib figure to an inline base64 PNG."""
    import io, base64
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=120)
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return f'<img src="data:image/png;base64,{encoded}" style="max-width:100%;"/>'

def _has_visible_content(fig):
    axes = fig.get_axes()
    if not axes:
        return False
    return any(
        ax.get_visible() and (
            ax.lines or ax.patches or ax.collections or ax.images or ax.texts
        )
        for ax in axes
    )

def capture_figures(fn, *args, **kwargs):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as _plt
    captured = []
    orig_show = _plt.show
    def _capture():
        captured.append(_plt.gcf())
        _plt.figure()
    _plt.show = _capture
    fn(*args, **kwargs)
    _plt.show = orig_show
    return [f for f in captured if _has_visible_content(f)]  # <-- updated

def filter_to_doc(annotations_dict, doc_id):
    """Return a dict scoped to a single document, or the full dict if doc_id is None."""
    if doc_id is None:
        return annotations_dict
    if doc_id not in annotations_dict:
        raise KeyError(f"Document ID {doc_id} not found in annotations.")
    return {doc_id: annotations_dict[doc_id]}

def filter_processed(processed_data, doc_id):
    """Filter the flat processed_data list to a single document, or return all if None."""
    if doc_id is None:
        return processed_data
    return [a for a in processed_data if a['task_id'] == doc_id]

def doc_suffix(doc_id):
    return f" — Document {doc_id}" if doc_id is not None else " — All Documents"

def completeness_to_html(df, span_desc=None, doc_desc=None):
    """Render a completeness DataFrame as two styled HTML tables."""
    if df is None or df.empty:
        return "<p><em>No completeness data available.</em></p>"

    span_cols = ['task_id', 'annotator_id', 'total_spans',
                 'subcategory_%', 'impact_%', 'span_comments_%']
    doc_cols  = ['task_id', 'annotator_id',
                 'has_document_issues',
                 'has_correspondence_score', 'has_correspondence_comment',
                 'has_readability_score',    'has_readability_comment']

    def df_to_table(sub_df, cols, caption, desc=None):
            tbl = sub_df[cols].copy()
            tbl.columns = [
                c.replace('has_', '')
                .replace('correspondence', 'correspond')
                .replace('readability', 'readable')
                for c in tbl.columns
            ]
            renamed_cols = list(tbl.columns)
            for c in renamed_cols:
                if tbl[c].dtype == bool:
                    tbl[c] = tbl[c].map({True: "✓", False: "✗"})
            header = "".join(f"<th>{c}</th>" for c in renamed_cols)
            rows_html = ""
            for _, row in tbl.iterrows():
                cells = ""
                for c in renamed_cols:
                    val = row[c]
                    style = ""
                    if val == "✗":
                        style = ' style="color:#c0392b; font-weight:bold;"'
                    elif isinstance(val, float) and val < 80:
                        style = ' style="color:#e67e22; font-weight:bold;"'
                    cells += f"<td{style}>{val}</td>"
                rows_html += f"<tr>{cells}</tr>"
            desc_html = f'<p style="color:#555; font-style:italic; font-size:.92rem; margin:.3rem 0 .7rem 0;">{desc}</p>' if desc else ""
            return f"""
            <h3 style="margin-top:1.2rem;">{caption}</h3>
            {desc_html}
            <div class="table-wrap">
            <table style="width:100%;">
            <thead><tr style="background:#f0f0f0;">{header}</tr></thead>
            <tbody>{rows_html}</tbody>
            </table>
            </div>"""

    span_tbl = df_to_table(df, span_cols, "Span-level Completeness (% filled)", desc=span_desc)
    doc_tbl  = df_to_table(df, doc_cols,  "Document-level Assessment Completeness", desc=doc_desc)
    return span_tbl + doc_tbl

# ── 1. Annotator completeness ──────────────────────────────────────────────────
if EXPORT["annotator_completeness"]:
    _doc_id  = EXPORT["annotator_completeness_doc"]
    _comp_df = summarize_annotation_completeness(task_id_to_annotations, doc_id=_doc_id)
    sections_html.append(section(
        f"Annotator Completeness{doc_suffix(_doc_id)}",
        completeness_to_html(
            _comp_df,
            span_desc=("Percentage of annotated spans where each annotator filled in the subcategory, "
                       "impact level, and span-level comment fields. Values below 80% are highlighted in orange."),
            doc_desc=("Whether each annotator completed all five document-level fields: "
                      "document issues, correspondence score, correspondence comment, readability score, and readability comment. "
                      "Missing fields are shown in red.")
        )
    ))

# ── 2. Annotator timing ────────────────────────────────────────────────────────
if EXPORT["annotator_timing"]:
    _doc_id = EXPORT["annotator_timing_doc"]
    _data   = filter_processed(processed_data, _doc_id)
    _df     = pd.DataFrame(_data)
    
    def _timing_stats_html(series, label, desc=None):
        data = series.dropna()
        if len(data) == 0:
            return f"<div><p><em>No {label} data available.</em></p></div>"
        q1, q3 = data.quantile(0.25), data.quantile(0.75)
        iqr = q3 - q1
        upper = q3 + 1.5 * iqr
        outliers = data[data > upper]
        outlier_html = ""
        if len(outliers):
            vals = ", ".join(f"{x:.1f}" for x in sorted(outliers))
            outlier_html = f"<tr><td>Outliers (&gt; {upper:.1f} min)</td><td>{len(outliers)} value(s): {vals}</td></tr>"
        desc_html = f'<p style="color:#555; font-style:italic; font-size:.92rem; margin:.3rem 0 .7rem 0;">{desc}</p>' if desc else ""
        return f"""
        <div>
        <h3 style="margin-top:1.2rem;">{label}</h3>
        {desc_html}
        <div class="table-wrap">
        <table style="width:100%;">
            <thead><tr style="background:#f0f0f0;"><th>Statistic</th><th>Value</th></tr></thead>
            <tbody>
            <tr><td>Measurements</td><td>{len(data)}</td></tr>
            <tr><td>Median (min)</td><td>{data.median():.1f}</td></tr>
            <tr><td>Mean (min)</td><td>{data.mean():.1f}</td></tr>
            <tr><td>Q1 – Q3 (min)</td><td>{q1:.1f} – {q3:.1f}</td></tr>
            <tr><td>Min (min)</td><td>{data.min():.1f}</td></tr>
            <tr><td>Max (min)</td><td>{data.max():.1f}</td></tr>
            {outlier_html}
            </tbody>
        </table>
        </div>
        </div>"""
    
    timing_html = (
        f'<div style="display:grid; grid-template-columns:1fr 1fr; gap:1.5rem; align-items:start;">'
        + _timing_stats_html(
            _df['lead_time_minutes'],
            "Lead Time",
            desc="Active time spent working on the task as recorded by Label Studio, "
                "excluding time the task was open but idle."
        )
        + _timing_stats_html(
            _df['review_time_minutes'],
            "Review Time",
            desc="Total elapsed time from when the annotator first opened the task to submission, "
                "which may include breaks or interruptions."
        )
        + '</div>'
    )
    sections_html.append(section(
        f"Annotator Timing{doc_suffix(_doc_id)}",
        timing_html,
        description="Outliers in both measures are identified using the 1.5×IQR rule."
    ))

# ── 3. Span overlap visualization ─────────────────────────────────────────────
if EXPORT["span_overlap_visualization"]:
    _doc_id = EXPORT["span_overlap_visualization_doc"]
    html = visualize_all_documents(task_id_to_annotations, selected_doc_id=_doc_id)
    sections_html.append(section(
        f"Annotation Overlap Visualization{doc_suffix(_doc_id)}",
        html,
        description="Each document is shown with annotated spans highlighted. "
                    "Darker shading indicates higher annotator agreement on a given span."
    ))

# ── 4. Error type distribution ─────────────────────────────────────────────────
if EXPORT["error_type_distribution"]:
    _doc_id = EXPORT["error_type_distribution_doc"]
    _data   = filter_to_doc(task_id_to_annotations, _doc_id)
    figs    = capture_figures(visualize_error_types, _data, include_subcategories=True)
    # Drop the "by document" chart when there's only one document
    if len(_data) == 1:
        figs = [f for f in figs if "by Document" not in f.axes[0].get_title()]
    imgs_html = "".join(fig_to_html(f) for f in figs)
    sections_html.append(section(
        f"Error Type Distribution{doc_suffix(_doc_id)}",
        imgs_html,
        description="Distribution of error labels, subcategories, and impact ratings "
                    + (f"for document {_doc_id}." if _doc_id is not None
                       else "across all documents in the project.")
    ))

# ── 5. Rating distributions ────────────────────────────────────────────────────
if EXPORT["rating_distributions"]:
    _doc_id = EXPORT["rating_distributions_doc"]
    _data   = filter_processed(processed_data, _doc_id)
    figs    = capture_figures(visualize_rating_distributions, _data)
    fig  = figs[0]
    axes = fig.get_axes()
    if len(axes) == 2:
        axes[1].set_visible(False)
        axes[0].set_position([0.1, 0.15, 0.75, 0.7])
        fig.set_size_inches(fig.get_figwidth(), fig.get_figheight() / 1.5)
        # Re-center suptitle over the plot area rather than the full figure
        fig.suptitle(fig.texts[0].get_text(), x=0.475)
    imgs_html = "".join(fig_to_html(f) for f in figs)
    sections_html.append(section(
        f"Correspondence & Readability Ratings{doc_suffix(_doc_id)}",
        imgs_html,
        description="Distribution of annotator ratings on overall translation correspondence (accuracy) "
                    "and readability, on a 1–4 scale."
    ))

# ── 6 & 7. Span matching — Exact + F1 (combined section) ─────────────────────
if EXPORT["exact_span_matching"] or EXPORT["span_f1"]:
    _doc_id       = EXPORT["exact_span_matching_doc"] or EXPORT["span_f1_doc"]
    _data         = filter_to_doc(task_id_to_annotations, _doc_id)
    exact_results = calculate_exact_matches(_data)
    f1_results    = calculate_span_f1(_data)
    figs          = capture_figures(display_span_f1_results, f1_results)
    imgs_html     = "".join(fig_to_html(f) for f in figs)
    combined_desc = (
        "<strong>Exact Match Rate</strong> — percentage of error spans where two or more annotators "
        "identified identical start and end character positions. Strictest measure of boundary agreement. "
        "<br><br>"
        "<strong>Precision</strong> — of all characters annotator A marked as errors, what proportion "
        "was also marked by annotator B. High precision means annotators rarely flag spans the other ignores. "
        "<br><br>"
        "<strong>Recall</strong> — of all characters annotator B marked as errors, what proportion "
        "was also marked by annotator A. High recall means annotators rarely miss spans the other caught. "
        "<br><br>"
        "<strong>F1 Score</strong> — harmonic mean of Precision and Recall, balancing the two into a single "
        "partial-overlap agreement score. Penalises large imbalances between precision and recall more than "
        "a simple average would. Unlike exact matching, partial credit is awarded when spans overlap, "
        "making F1 a more lenient measure of boundary agreement. "
        "Production deployment target for F1: &#x2265;0.70."
    )
    sections_html.append(section(
        f"Span Agreement — Exact Match, F1, Precision &amp; Recall{doc_suffix(_doc_id)}",
        imgs_html,
        description=combined_desc
    ))

# ── 8. Cohen's Kappa – categories ─────────────────────────────────────────────
if EXPORT["cohens_kappa_categories"]:
    _doc_id      = EXPORT["cohens_kappa_categories_doc"]
    _data        = filter_to_doc(task_id_to_annotations, _doc_id)
    cat_agreement = calculate_category_agreement(_data, attribute='labels')
    figs          = capture_figures(display_category_agreement, cat_agreement, "Category")
    imgs_html     = "".join(fig_to_html(f) for f in figs)
    sections_html.append(section(
        f"Cohen's Kappa — Error Category Agreement{doc_suffix(_doc_id)}",
        imgs_html,
        description="Pairwise Cohen's Kappa scores measuring agreement on error category labels "
                    "between each pair of annotators, accounting for chance agreement. "
                    "Cohen&#8217;s &#954; measures agreement on error category labels for overlapping spans. "
                    "Production deployment target for both metrics: &#x2265;0.60. "
                    "Low reliability in one condition means its quality metrics should be interpreted with caution. "
                    "Kappa values: &lt;0.20 poor · 0.21&#8211;0.40 fair · 0.41&#8211;0.60 moderate · "
                    "0.61&#8211;0.80 substantial · &gt;0.80 almost perfect."
    ))

# ── Assemble and write HTML ────────────────────────────────────────────────────
html_doc = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{PROJECT_NAME} – Inter-Rater Reliability Report – {REPORT_DATE}</title>
  <style>
    body        {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
                   max-width: 1100px; margin: 2rem auto; padding: 0 1.5rem;
                   color: #222; line-height: 1.55; }}
    h1          {{ border-bottom: 3px solid #444; padding-bottom: .5rem; }}
    h2          {{ color: #333; margin-top: .5rem; }}
    h3          {{ color: #555; margin-top: 0rem; }}
    .meta       {{ color: #666; font-size: .9rem; margin-bottom: 2rem; }}
    .section-desc {{ color: #555; font-style: italic; font-size: .92rem;
                     margin: .4rem 0 1rem 0; }}
    section     {{ background: #fafafa; border: 1px solid #e0e0e0;
                   border-radius: 6px; padding: 1.2rem 1.5rem; }}
    .table-wrap {{ overflow-x: auto; margin-top: .5rem; }}
    table       {{ border-collapse: collapse; white-space: nowrap; }}
    th, td      {{ border: 1px solid #ddd; padding: .35rem .7rem; text-align: left; }}
    thead tr    {{ background: #f0f0f0; }}
  </style>
</head>
<body>
  <h1>{PROJECT_NAME} – Inter-Rater Reliability Report</h1>
  <p class="meta">Round date: {REPORT_DATE} &nbsp;|&nbsp;
     Annotators: {sum(len(v) for v in task_id_to_annotations.values())} annotations across
     {len(task_id_to_annotations)} task(s)</p>
  {chr(10).join(sections_html)}
</body>
</html>"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(html_doc)

print(f"Report written to: {report_path}")
display(HTML(f'<b>Report saved:</b> <code>{report_path}</code>'))

SPAN-LEVEL COMPLETENESS (All Documents)
% of spans with field filled in

 task_id  annotator_id  total_spans  subcategory_%  impact_%  span_comments_%
      32            58           49           93.9      98.0             85.7
      32            61           34          100.0     100.0            100.0

Annotators below 80% on any span field:
  None ✓


DOCUMENT-LEVEL COMPLETENESS (All Documents)

 task_id  annotator_id  has_document_issues  has_correspondence_score  has_correspondence_comment  has_readability_score  has_readability_comment
      32            58                 True                      True                        True                   True                     True
      32            61                 True                      True                       False                   True                    False

Annotators with incomplete document-level assessment:
 task_id  annotator_id  has_document_issues  has_correspondence_score  has_correspondence_comment  has_